## Hotel Booking Prediction
author: Virginia Ordoño Bernier
date: April 2024

### Descripción
- Hay que construir un modelo de red neuronal que permita predecir si una reserva de Booking será o no cancelada.
- El dataset contiene datos sobre la reserva de un hotel urbano y de un hotel turístico, e incluye información como la fecha de la reserva, la duración de la estancia, el número de adultos, niños y/o bebés, y el número de plazas de aparcamiento disponibles, entre otras variables.
- Hay que realizar las transformaciones pertinentes en base a un estudio previo del dataset donde, entre otras cosas, hay que comprobar si las clases están o no balanceadas y se requiere resampling, si todas las columnas son necesarias, si hay valores erróneos o nulos, si hay valores categóricos que deben transformarse, etc...

### Carga datos

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv('./data/bookings.csv')

df.head(5)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [51]:
import pandas as pd

# Supongamos que df_booking es tu DataFrame existente
# Aquí obtendremos los nombres de las columnas de df_booking
df_names = df.columns.tolist()

# Lista de traducciones correspondientes a los nombres de las columnas
traducciones = [
    'hotel', 'está_cancelado', 'tiempo_previo', 'año_de_llegada',
    'mes_de_llegada', 'número_de_semana_de_llegada',
    'día_del_mes_de_llegada', 'noches_de_fin_de_semana',
    'noches_de_semana', 'adultos', 'niños', 'bebés', 'comida',
    'país', 'segmento_de_mercado', 'canal_de_distribución',
    'es_huésped_repetido', 'cancelaciones_previas',
    'reservas_previas_no_canceladas', 'tipo_de_habitación_reservada',
    'tipo_de_habitación_asignada', 'cambios_en_la_reserva', 'tipo_de_deposito', 'agente',
    'empresa', 'días_en_lista_de_espera', 'tipo_de_cliente', 'tarifa_diaria_promedio',
    'espacios_de_estacionamiento_requeridos', 'total_de_solicitudes_especiales',
    'estado_de_reserva', 'fecha_de_estado_de_reserva'
]

# Crear un nuevo DataFrame con los nombres de las columnas y sus traducciones
df_names_translated = pd.DataFrame({
    'Columna': df_names,
    'Traducción': traducciones
})

# Mostrar el DataFrame resultante
print(df_names_translated)


                           Columna                              Traducción
0                            hotel                                   hotel
1                      is_canceled                          está_cancelado
2                        lead_time                           tiempo_previo
3                arrival_date_year                          año_de_llegada
4               arrival_date_month                          mes_de_llegada
5         arrival_date_week_number             número_de_semana_de_llegada
6        arrival_date_day_of_month                  día_del_mes_de_llegada
7          stays_in_weekend_nights                 noches_de_fin_de_semana
8             stays_in_week_nights                        noches_de_semana
9                           adults                                 adultos
10                        children                                   niños
11                          babies                                   bebés
12                       

In [52]:
df.shape

(119390, 32)

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

### Detección de NaN por columna

In [54]:
# Rows = 119390
if (df.isna().sum() > 0).any():
    print(df.isna().sum()[df.isna().sum() > 0])

children         4
country        488
agent        16340
company     112593
dtype: int64


In [55]:
# Los NaN de 'children' los ponemos a 0 ya que son pocos y damos por hecho que no llevan niños
df['children'] = df['children'].fillna(0)

# Los NaN de 'country' los ponemos a 'Unknown' ya que son pocos y no sabemos de donde son
df['country'] = df['country'].fillna('Unknown')

# Los NaN de 'agent' los ponemos a 0 ya que son pocos y damos por hecho que no tienen agente
df['agent'] = df['agent'].fillna(0)

# Eliminamos la columna 'company' ya que tiene muchos NaN y pensamos que no aporta información relevante
df = df.drop(columns=['company'])   

if (df.isna().sum() > 0).any():
    print(df.isna().sum()[df.isna().sum() > 0])

### Detección de duplicados

In [56]:
if df.duplicated().any():
    duplicated_rows = df[df.duplicated(keep=False)]  # keep=False para marcar todas las filas duplicadas
    print(duplicated_rows)

In [ ]:
# Eliminamos las filas duplicadas ya que entendemos que pueden suponer un error en la predicción
df.drop_duplicates(inplace=True)

### Separación tipología de datos

In [ ]:
categorical_columns_names = df.select_dtypes(include=['object']).columns.tolist()
numeric_columns_names = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
all_columns_names = categorical_columns_names + numeric_columns_names
target = 'is_canceled'


### Normalización daros numéricos

### Comprobacación de la cardinalidad

In [60]:
categ_columns = df.select_dtypes(include=['object', 'category']).columns

for column in categ_columns:
    print(f"'{column}': {df[column].unique()}")

'hotel': ['Resort Hotel' 'City Hotel']
'arrival_date_month': ['July' 'August' 'September' 'October' 'November' 'December' 'January'
 'February' 'March' 'April' 'May' 'June']
'meal': ['BB' 'FB' 'HB' 'SC' 'Undefined']
'country': ['PRT' 'GBR' 'USA' 'ESP' 'IRL' 'FRA' 'Unknown' 'ROU' 'NOR' 'OMN' 'ARG'
 'POL' 'DEU' 'BEL' 'CHE' 'CN' 'GRC' 'ITA' 'NLD' 'DNK' 'RUS' 'SWE' 'AUS'
 'EST' 'CZE' 'BRA' 'FIN' 'MOZ' 'BWA' 'LUX' 'SVN' 'ALB' 'IND' 'CHN' 'MEX'
 'MAR' 'UKR' 'SMR' 'LVA' 'PRI' 'SRB' 'CHL' 'AUT' 'BLR' 'LTU' 'TUR' 'ZAF'
 'AGO' 'ISR' 'CYM' 'ZMB' 'CPV' 'ZWE' 'DZA' 'KOR' 'CRI' 'HUN' 'ARE' 'TUN'
 'JAM' 'HRV' 'HKG' 'IRN' 'GEO' 'AND' 'GIB' 'URY' 'JEY' 'CAF' 'CYP' 'COL'
 'GGY' 'KWT' 'NGA' 'MDV' 'VEN' 'SVK' 'FJI' 'KAZ' 'PAK' 'IDN' 'LBN' 'PHL'
 'SEN' 'SYC' 'AZE' 'BHR' 'NZL' 'THA' 'DOM' 'MKD' 'MYS' 'ARM' 'JPN' 'LKA'
 'CUB' 'CMR' 'BIH' 'MUS' 'COM' 'SUR' 'UGA' 'BGR' 'CIV' 'JOR' 'SYR' 'SGP'
 'BDI' 'SAU' 'VNM' 'PLW' 'QAT' 'EGY' 'PER' 'MLT' 'MWI' 'ECU' 'MDG' 'ISL'
 'UZB' 'NPL' 'BHS' 'MAC' 'TGO' 'TWN' 'DJI' 'ST